In [1]:
import pandas as pd
import numpy as np
import os, gc
import pyarrow.parquet as pq
import pyarrow as pa

MASTER_PATH = "/kaggle/input/datasets/ayastudentkhoumari/algeria-trade/algeria_trade_master.parquet"
OUT = "/kaggle/working/algeria_trade_master_fe.parquet"
ALGERIA = "DZA"

# check years
years_df = pq.read_table(MASTER_PATH, columns=["t"]).to_pandas()
YEARS = sorted(years_df["t"].unique().tolist())
del years_df
gc.collect()
print(f"Years: {YEARS}")
print(f"Total years: {len(YEARS)}")

Years: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2020, 2021, 2022, 2023, 2024]
Total years: 14


#  Pass 1: compute global aggregates per (year, product)

In [2]:
# We need these for every row later:
# - world_imports: total world imports of product k in year t
# - global_demand_index: normalized version of world_imports
# - algeria_exports: total Algeria exports of product k in year t (for market_share)

world_agg_list = []
hhi_list = []
alg_exp_list = []

for year in YEARS:
    print(f"  year {year}...", end=" ")

    yr = pq.read_table(
        MASTER_PATH,
        columns=["t", "k", "iso3_i", "iso3_j", "v"],
        filters=[("t", "=", year)]
    ).to_pandas()

    yr["k"] = yr["k"].astype(str)
    yr["v"] = yr["v"].astype("float32")

    # world_imports = total value all countries import per product
    wi = (
        yr.groupby("k")["v"]
        .sum()
        .reset_index()
        .rename(columns={"v": "world_imports"})
    )
    wi["t"] = year
    wi["world_imports"] = wi["world_imports"].astype("float32")
    world_agg_list.append(wi)

    # HHI = export concentration per product
    exp = yr.groupby(["k", "iso3_i"])["v"].sum().reset_index()
    exp = exp.merge(wi[["k", "world_imports"]], on="k", how="left")
    exp["share"] = (exp["v"] / exp["world_imports"].replace(0, np.nan)).astype("float32")
    hhi_yr = (
        exp.groupby("k")["share"]
        .apply(lambda s: float((s**2).sum()))
        .reset_index()
    )
    hhi_yr.columns = ["k", "hhi"]
    hhi_yr["t"] = year
    hhi_yr["hhi"] = hhi_yr["hhi"].astype("float32")
    hhi_list.append(hhi_yr)

    # Algeria exports per product
    alg = yr[yr["iso3_i"] == ALGERIA].groupby("k")["v"].sum().reset_index()
    alg.columns = ["k", "algeria_exports"]
    alg["t"] = year
    alg["algeria_exports"] = alg["algeria_exports"].astype("float32")
    alg_exp_list.append(alg)

    print(f"{len(yr):,} rows")
    del yr, wi, exp, hhi_yr, alg
    gc.collect()

world_agg = pd.concat(world_agg_list, ignore_index=True)
hhi_df    = pd.concat(hhi_list,       ignore_index=True)
alg_exp   = pd.concat(alg_exp_list,   ignore_index=True)
del world_agg_list, hhi_list, alg_exp_list
gc.collect()

# global demand index = normalized world_imports within each year
world_agg["global_demand_index"] = (
    world_agg.groupby("t")["world_imports"]
    .transform(lambda x: (x / x.max()).astype("float32"))
)

print(f"\nworld_agg : {world_agg.shape}")
print(f"hhi_df    : {hhi_df.shape}")
print(f"alg_exp   : {alg_exp.shape}")

  year 2010... 9,470,765 rows
  year 2011... 9,640,712 rows
  year 2012... 9,937,050 rows
  year 2013... 10,098,942 rows
  year 2014... 10,145,672 rows
  year 2015... 10,499,686 rows
  year 2016... 10,569,571 rows
  year 2017... 10,821,738 rows
  year 2018... 10,924,964 rows
  year 2020... 10,638,241 rows
  year 2021... 11,146,476 rows
  year 2022... 11,154,874 rows
  year 2023... 11,194,668 rows
  year 2024... 10,580,049 rows

world_agg : (67154, 4)
hhi_df    : (67154, 3)
alg_exp   : (24421, 3)


# Pass 2: build full panel year by year and write incrementally

In [3]:
KEEP_COLS = [
    "t", "k", "iso3_i", "iso3_j", "v", "q",
    "product_description",
    "gdp_j", "gdp_growth_j", "population_j",
    "trade_percent_gdp_j", "inflation_j",
    "dist", "contig", "comlang_off", "colony",
    "continent_j", "landlocked_j"
]

writer = None

for year in YEARS:
    print(f"  year {year}...", end=" ")

    yr = pq.read_table(
        MASTER_PATH,
        columns=KEEP_COLS,
        filters=[("t", "=", year)]
    ).to_pandas()

    yr["k"] = yr["k"].astype(str)
    for col in yr.select_dtypes("float64").columns:
        yr[col] = yr[col].astype("float32")

    yr = yr.rename(columns={
        "t":                   "year",
        "k":                   "product_code",
        "iso3_i":              "exporter",
        "iso3_j":              "partner",
        "v":                   "fobvalue",
        "q":                   "qty",
        "gdp_j":               "gdp",
        "gdp_growth_j":        "gdp_growth",
        "population_j":        "population",
        "trade_percent_gdp_j": "trade_percent_gdp",
        "inflation_j":         "inflation",
    })

    # global_imports = total imports of this product by this partner (from all exporters)
    global_imp = (
        yr.groupby(["product_code", "partner"])["fobvalue"]
        .sum()
        .reset_index()
        .rename(columns={"fobvalue": "global_imports"})
    )
    global_imp["global_imports"] = global_imp["global_imports"].astype("float32")
    yr = yr.merge(global_imp, on=["product_code", "partner"], how="left")

    # join world_imports, global_demand_index
    wa_yr = world_agg[world_agg["t"] == year][
        ["k", "world_imports", "global_demand_index"]
    ].rename(columns={"k": "product_code"})
    yr = yr.merge(wa_yr, on="product_code", how="left")

    # join hhi
    hhi_yr = hhi_df[hhi_df["t"] == year][["k", "hhi"]].rename(columns={"k": "product_code"})
    yr = yr.merge(hhi_yr, on="product_code", how="left")

    # join algeria_exports
    alg_yr = alg_exp[alg_exp["t"] == year][["k", "algeria_exports"]].rename(columns={"k": "product_code"})
    yr = yr.merge(alg_yr, on="product_code", how="left")
    yr["algeria_exports"] = yr["algeria_exports"].fillna(0).astype("float32")

    # ── engineered features ──────────────────────────────────────────

    # market_share = this exporter's share of world exports for this product
    exporter_total = (
        yr.groupby(["exporter", "product_code"])["fobvalue"]
        .sum()
        .reset_index()
        .rename(columns={"fobvalue": "exporter_total"})
    )
    yr = yr.merge(exporter_total, on=["exporter", "product_code"], how="left")
    yr["market_share"] = (
        yr["exporter_total"] / yr["world_imports"].replace(0, np.nan)
    ).astype("float32")
    yr.drop(columns=["exporter_total"], inplace=True)

    # penetration_ratio = fobvalue / global_imports (how much of partner's imports this exporter covers)
    yr["penetration_ratio"] = (
        yr["fobvalue"] / yr["global_imports"].replace(0, np.nan)
    ).clip(0, 1).astype("float32")

    # trade_balance = exports - imports for this exporter-partner pair
    # (approximated as fobvalue since we only have export direction here)
    yr["trade_balance"] = yr["fobvalue"].astype("float32")

    # product_diversity = how many distinct products this exporter sends to this partner
    div = (
        yr.groupby(["exporter", "partner"])["product_code"]
        .nunique()
        .reset_index()
        .rename(columns={"product_code": "product_diversity"})
    )
    yr = yr.merge(div, on=["exporter", "partner"], how="left")
    yr["product_diversity"] = yr["product_diversity"].astype("int16")

    # export_growth = 0 for now (fixed in next cell using full panel)
    yr["export_growth"] = np.float32(0)

    # ── opportunity score & label ────────────────────────────────────
    def norm(s):
        mn, mx = s.min(), s.max()
        return ((s - mn) / (mx - mn + 1e-8)).astype("float32")

    yr["opportunity_score"] = (
        0.30 * norm(yr["global_demand_index"].fillna(0)) +
        0.25 * (1 - norm(yr["market_share"].fillna(0)))  +
        0.20 * norm(yr["gdp"].fillna(0))                 +
        0.15 * (1 - norm(yr["penetration_ratio"].fillna(0))) +
        0.10 * norm(yr["export_growth"].fillna(0))
    ).astype("float32")

    q33 = yr["opportunity_score"].quantile(0.33)
    q67 = yr["opportunity_score"].quantile(0.67)
    yr["opportunity_level"] = pd.cut(
        yr["opportunity_score"],
        bins=[-np.inf, q33, q67, np.inf],
        labels=["Low", "Medium", "High"]
    ).astype(str)

    # final dtype cleanup
    for col in yr.select_dtypes("float64").columns:
        yr[col] = yr[col].astype("float32")

    # write incrementally
    table = pa.Table.from_pandas(yr, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUT, table.schema, compression="snappy")
    try:
        writer.write_table(table)
    except Exception:
        writer.write_table(table.cast(writer.schema))

    print(f"{len(yr):,} rows written")
    del yr, global_imp, wa_yr, hhi_yr, alg_yr, div, table, exporter_total
    gc.collect()

writer.close()
print(f"\nSaved → {OUT}")
print(f"Size  : {os.path.getsize(OUT)/1e6:.1f} MB")

  year 2010... 9,470,765 rows written
  year 2011... 9,640,712 rows written
  year 2012... 9,937,050 rows written
  year 2013... 10,098,942 rows written
  year 2014... 10,145,672 rows written
  year 2015... 10,499,686 rows written
  year 2016... 10,569,571 rows written
  year 2017... 10,821,738 rows written
  year 2018... 10,924,964 rows written
  year 2020... 10,638,241 rows written
  year 2021... 11,146,476 rows written
  year 2022... 11,154,874 rows written
  year 2023... 11,194,668 rows written
  year 2024... 10,580,049 rows written

Saved → /kaggle/working/algeria_trade_master_fe.parquet
Size  : 5184.0 MB


# Fix export_growth using full panel

In [4]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import numpy as np
import os, gc

OUT       = "/kaggle/working/algeria_trade_master_fe.parquet"
OUT_FIXED = "/kaggle/working/algeria_trade_master_fe_fixed.parquet"

pf = pq.ParquetFile(OUT)

# get years safely without loading full file
meta = pq.read_table(OUT, columns=["year"]).to_pandas()
YEARS = sorted(meta["year"].unique().tolist())
del meta
gc.collect()

writer = None

# previous year lookup
prev_year_lookup = None

print(f"Processing {len(YEARS)} years...\n")

for year in YEARS:

    print(f"  processing year {year}...", end=" ")

    # read only current year
    yr = pq.read_table(
        OUT,
        filters=[("year", "=", year)]
    ).to_pandas()

    # remove old placeholder column
    yr = yr.drop(columns=["export_growth"], errors="ignore")

    # keep only columns needed for growth computation
    curr_lookup = yr[[
        "exporter",
        "product_code",
        "partner",
        "fobvalue"
    ]].copy()

    curr_lookup["fobvalue"] = curr_lookup["fobvalue"].astype("float32")

    # first year → no previous growth
    if prev_year_lookup is None:

        yr["export_growth"] = np.float32(0)

    else:

        prev_lookup = prev_year_lookup.rename(
            columns={"fobvalue": "prev_fobvalue"}
        )

        growth = curr_lookup.merge(
            prev_lookup,
            on=["exporter", "product_code", "partner"],
            how="left"
        )

        growth["export_growth"] = (
            (
                growth["fobvalue"] -
                growth["prev_fobvalue"]
            )
            /
            growth["prev_fobvalue"].replace(0, np.nan)
        )

        growth["export_growth"] = (
            growth["export_growth"]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .clip(-5, 5)
            .astype("float32")
        )

        yr["export_growth"] = growth["export_growth"]

        del growth, prev_lookup
        gc.collect()

    # save current lookup for next iteration
    prev_year_lookup = curr_lookup.copy()

    # recompute opportunity score
    def norm(s):
        mn, mx = s.min(), s.max()
        return ((s - mn) / (mx - mn + 1e-8)).astype("float32")

    yr["opportunity_score"] = (
        0.30 * norm(yr["global_demand_index"].fillna(0)) +
        0.25 * (1 - norm(yr["market_share"].fillna(0))) +
        0.20 * norm(yr["gdp"].fillna(0)) +
        0.15 * (1 - norm(yr["penetration_ratio"].fillna(0))) +
        0.10 * norm(yr["export_growth"].fillna(0))
    ).astype("float32")

    q33 = yr["opportunity_score"].quantile(0.33)
    q67 = yr["opportunity_score"].quantile(0.67)

    yr["opportunity_level"] = pd.cut(
        yr["opportunity_score"],
        bins=[-np.inf, q33, q67, np.inf],
        labels=["Low", "Medium", "High"]
    ).astype(str)

    # optimize dtypes
    for col in yr.select_dtypes("float64").columns:
        yr[col] = yr[col].astype("float32")

    table = pa.Table.from_pandas(
        yr,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            OUT_FIXED,
            table.schema,
            compression="snappy"
        )

    try:
        writer.write_table(table)

    except Exception:
        writer.write_table(table.cast(writer.schema))

    print(f"{len(yr):,} rows")

    del yr, curr_lookup, table
    gc.collect()

writer.close()

# replace old file
os.remove(OUT)
os.rename(OUT_FIXED, OUT)

print("\nDONE")
print(f"Final file: {OUT}")
print(f"Size: {os.path.getsize(OUT)/1e6:.1f} MB")

Processing 14 years...

  processing year 2010... 9,470,765 rows
  processing year 2011... 9,640,712 rows
  processing year 2012... 9,937,050 rows
  processing year 2013... 10,098,942 rows
  processing year 2014... 10,145,672 rows
  processing year 2015... 10,499,686 rows
  processing year 2016... 10,569,571 rows
  processing year 2017... 10,821,738 rows
  processing year 2018... 10,924,964 rows
  processing year 2020... 10,638,241 rows
  processing year 2021... 11,146,476 rows
  processing year 2022... 11,154,874 rows
  processing year 2023... 11,194,668 rows
  processing year 2024... 10,580,049 rows

DONE
Final file: /kaggle/working/algeria_trade_master_fe.parquet
Size: 5705.0 MB


# Verifying

In [5]:
import pyarrow.parquet as pq
import gc

# open parquet metadata only
pf = pq.ParquetFile(OUT)

print(f"Number of row groups : {pf.num_row_groups}")
print(f"Total rows           : {pf.metadata.num_rows:,}")
print(f"Total columns        : {pf.metadata.num_columns}")

print("\nColumns:")
for i, col in enumerate(pf.schema.names, 1):
    print(f"  {i:2}. {col}")

# read ONLY a tiny sample
sample = pf.read_row_group(0).to_pandas().head(5)

print("\nSample:")
print(sample[[
    "year",
    "exporter",
    "partner",
    "product_code",
    "fobvalue",
    "market_share",
    "opportunity_level"
]])

del sample
gc.collect()

print("\nDONE — download algeria_trade_master_fe.parquet from Output tab")

Number of row groups : 149
Total rows           : 146,823,408
Total columns        : 30

Columns:
   1. year
   2. product_code
   3. exporter
   4. partner
   5. fobvalue
   6. qty
   7. product_description
   8. gdp
   9. gdp_growth
  10. population
  11. trade_percent_gdp
  12. inflation
  13. dist
  14. contig
  15. comlang_off
  16. colony
  17. continent_j
  18. landlocked_j
  19. global_imports
  20. world_imports
  21. global_demand_index
  22. hhi
  23. algeria_exports
  24. market_share
  25. penetration_ratio
  26. trade_balance
  27. product_diversity
  28. opportunity_score
  29. opportunity_level
  30. export_growth

Sample:
   year exporter partner product_code  fobvalue  market_share  \
0  2010      AFG     ALB       610690     0.192  4.290042e-07   
1  2010      AFG     ALB       621590     0.845  1.599126e-05   
2  2010      AFG     ALB       630900     0.317  3.589011e-05   
3  2010      AFG     ALB       851999     0.150  6.395827e-06   
4  2010      AFG     ALB    